In [14]:
import json
from pathlib import Path
import psycopg2
from dotenv import load_dotenv
import os

# Load environment variables from .env (if you use dotenv)
load_dotenv()

# Database connection parameters
db_params = {
    'dbname': os.getenv('POSTGRES_DB', 'ethiomed_db'),   # fallback to your db name
    'user': os.getenv('POSTGRES_USER', 'ethiomed_user'), 
    'password': os.getenv('POSTGRES_PASSWORD', 'ethiomed_pass'),
    'host': os.getenv('POSTGRES_HOST', 'localhost'),
    'port': os.getenv('POSTGRES_PORT', '5432')
}

def load_json_to_postgres(json_path, channel, conn):
    with open(json_path, 'r', encoding='utf-8') as f:
        messages = json.load(f)

    cur = conn.cursor()
    for msg in messages:
        # msg might have 'message_id' or 'id', check your JSON keys
        msg_id = msg.get('message_id') or msg.get('id')
        date = msg.get('date')
        text = msg.get('text')
        has_image = msg.get('has_image', False)  # default False if missing

        cur.execute("""
            INSERT INTO raw.telegram_messages (id, channel, date, text, has_image)
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (id) DO NOTHING;
        """, (msg_id, channel, date, text, has_image))
    conn.commit()
    cur.close()

# Connect to PostgreSQL
conn = psycopg2.connect(**db_params)

# Path to the directory with JSON files
data_dir = Path(r'C:\Users\yusuf\Desktop\10Academy\week 7\kaim_week7\notebooks\data\raw\telegram_messages\2025-07-14')

# Iterate over all json files in the directory
for json_file in data_dir.glob('*.json'):
    channel_name = json_file.stem  # file name without .json
    print(f"Loading data for channel: {channel_name}")
    load_json_to_postgres(json_file, channel_name, conn)

conn.close()

print("All data loaded successfully.")


Loading data for channel: CheMed123
Loading data for channel: lobelia4cosmetics
Loading data for channel: tikvahpharma
All data loaded successfully.
